# Task 1.1: Core Contribution / Architecture## Paper: Clustering Time Series Using Unsupervised-Shapelets**Authors**: Jesin Zakaria, Abdullah Mueen, Eamonn J. Keogh  **Venue**: ICDM 2012 (IEEE International Conference on Data Mining)

## Step-by-Step Method Description### Step 1: Candidate Subsequence Generation- **Description**: A sliding window of length *l* is moved across every time series in the dataset *D* to extract all possible subsequences of that length. The algorithm considers a range of lengths from *l_min* to *l_max* (typically 1/20 to 1/5 of the time series length). Each extracted subsequence becomes a candidate u-shapelet.- **Reference**: Section III-A of the paper; the exhaustive search over subsequences is the outer loop of Algorithm 1.- **Purpose**: This step creates the pool of candidate patterns from which the algorithm will select the most discriminative ones. Unlike supervised shapelets that use class labels to prune candidates, here *every* subsequence is a potential candidate because no labels exist.

### Step 2: Subsequence Distance (sdist) Computation- **Description**: For a given candidate subsequence *S* of length *m* and a time series *T* of length *n*, the subsequence distance is computed as:  `sdist(S, T) = min_{1 ≤ i ≤ n-m+1} EuclideanDist(S, T[i:i+m])`  That is, *S* is slid along *T* and the minimum Euclidean distance across all alignments is returned. This is computed for *every* time series in the dataset, producing a vector of *N* distance values (one per time series) for each candidate.- **Reference**: Equation (1) in the paper (Definition 2: Subsequence Distance).- **Purpose**: sdist captures how similar a candidate pattern is to each time series, focusing on the best local match. This is the fundamental building block that makes the method local-pattern-aware rather than whole-series-aware.

### Step 3: Gap Metric Evaluation- **Description**: Given the vector of sdist values for a candidate, the algorithm sorts these distances and searches for the optimal split point that divides the dataset into two groups — *D_A* (close to the candidate) and *D_B* (far from the candidate). The quality of this split is measured by the **gap metric**:  `GAP(S) = (mean(D_B) − std(D_B)) − (mean(D_A) + std(D_A))`  A larger gap means the candidate produces a clearer bimodal separation in sdist values. All possible split points are evaluated (or a pruned set using early-abandon), and the one maximising the gap is selected.- **Reference**: Definition 4 (Gap Score) in Section III-B; visualised in Figure 3 of the paper.- **Purpose**: The gap metric replaces the information-gain criterion used in supervised shapelets. It quantifies how well a candidate can separate time series into two distinct groups *without labels*, using only the statistical separation of distance distributions.

### Step 4: Greedy U-Shapelet Selection with Iterative Separation- **Description**: The algorithm greedily selects the candidate with the *largest* gap score as the first u-shapelet. The group *D_A* (time series closest to this shapelet) is then removed from the dataset. The process repeats on the remaining time series *D_B*: new candidates are extracted, sdists recomputed, and the next best u-shapelet is selected. This continues until no candidate achieves a gap score above a threshold, or all time series have been assigned to some group.- **Reference**: Algorithm 1 in Section III-C; the iterative peeling procedure is the key novelty.- **Purpose**: This iterative approach avoids the need to pre-specify the number of clusters. Each u-shapelet peels off one coherent group, allowing the algorithm to discover the natural cluster structure. It also prevents a single dominant pattern from masking weaker but valid clusters.

### Step 5: Distance Map Construction- **Description**: After discovering *m* u-shapelets, the algorithm constructs an *N × m* distance map. Entry *(j, k)* = `sdist(u-shapelet_k, T_j)` — the subsequence distance between the *k*-th u-shapelet and the *j*-th time series. This transforms the original time series (which may differ in length and are hard to compare) into fixed-length vectors in an *m*-dimensional space.- **Reference**: Section III-D; Figure 5 illustrates the distance map for the CBF dataset.- **Purpose**: The distance map is the key representational innovation. It converts the clustering problem from the difficult time-series domain (where distance measures like DTW are expensive and noisy) into a standard Euclidean space where off-the-shelf clustering algorithms work well.

### Step 6: k-Means Clustering on the Distance Map- **Description**: Standard k-Means clustering is applied to the rows of the distance map (each row is an *m*-dimensional vector). The number of clusters *k* can be chosen based on prior knowledge or estimated from the number of non-trivial u-shapelets discovered. The final output is a cluster assignment for each time series.- **Reference**: Section III-D and the experimental evaluation in Section IV.- **Purpose**: By clustering in the shapelet-distance space, the algorithm achieves interpretable results — each cluster can be characterised by which u-shapelets its members are close to or far from.

## Final SummaryThis paper solves the problem of **unsupervised time series clustering** by introducing u-shapelets — discriminative subsequences discovered without labels using a gap metric — and the authors claim their approach is superior to existing alternatives (whole-series DTW/Euclidean clustering) because it focuses on local, interpretable patterns rather than entire time series, making it robust to noise, phase shifts, and irrelevant segments.